Import Libraries

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax" # you can also use tensorflow or torch

import keras_cv
import keras
from keras import ops
import tensorflow as tf

import cv2
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm
import joblib

import matplotlib.pyplot as plt 

In [ ]:
import tensorflow as tf

# Configurer la croissance de la mémoire GPU avant toute autre opération
physical_devices = tf.config.list_physical_devices('GPU')
if len(physical_devices) > 0:
    # Configurer pour permettre la croissance de la mémoire GPU
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
    print("GPU disponible et configuré")
else:
    print("Aucun GPU trouvé, utilisation du CPU.")


⚙️ | Configuration

In [ ]:
class CFG:
    verbose = 1  # Verbosity
    seed = 42  # Random seed
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [400, 300]  # Input image size
    epochs = 13 # Training epochs
    batch_size = 64  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    drop_remainder = True  # Drop incomplete batches
    num_classes = 6 # Number of classes in the dataset
    fold = 0 # Which fold to set as validation data
    class_names = ['Seizure', 'LPD', 'GPD', 'LRDA','GRDA', 'Other']
    label2name = dict(enumerate(class_names)) #Dictionary: index ➝ class name
    name2label = {v:k for k, v in label2name.items()} #Reverse of the previous: name ➝ index

♻️ | Reproducibility

In [ ]:
keras.utils.set_random_seed(CFG.seed)

Kaggle Config

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = ""
os.environ['KAGGLE_KEY'] = ""


Download the dataset from kaggle 

In [ ]:
!kaggle competitions download -c hms-harmful-brain-activity-classification


In [ ]:
!unzip -q hms-harmful-brain-activity-classification.zip -d ./hms_data


📁 | Dataset Path

In [ ]:
BASE_PATH = "./hms_data"  # Local path to extracted files


SPEC_DIR = "/tmp/dataset/hms-hbac"
os.makedirs(SPEC_DIR+'/train_spectrograms', exist_ok=True)
os.makedirs(SPEC_DIR+'/test_spectrograms', exist_ok=True)

In [ ]:
# Train + Valid
df = pd.read_csv(f'{BASE_PATH}/train.csv')
df['eeg_path'] = f'{BASE_PATH}/train_eegs/' + df['eeg_id'].astype(str) + '.parquet'
df['spec_path'] = f'{BASE_PATH}/train_spectrograms/' + df['spectrogram_id'].astype(str) + '.parquet'
df['spec2_path'] = f'{SPEC_DIR}/train_spectrograms/' + df['spectrogram_id'].astype(str) + '.npy'
df['class_name'] = df.expert_consensus.copy()
df['class_label'] = df.expert_consensus.map(CFG.name2label)
display(df.head(2))

# Test
test_df = pd.read_csv(f'{BASE_PATH}/test.csv')
test_df['eeg_path'] = f'{BASE_PATH}/test_eegs/' + test_df['eeg_id'].astype(str) + '.parquet'
test_df['spec_path'] = f'{BASE_PATH}/test_spectrograms/' + test_df['spectrogram_id'].astype(str) + '.parquet'
test_df['spec2_path'] = f'{SPEC_DIR}/test_spectrograms/' + test_df['spectrogram_id'].astype(str) + '.npy'
display(test_df.head(2))

Convert .parquet to .npy

To facilitate easier data loading, we will convert the EEG spectrograms from parquet to npy format. This process involves saving the spectrogram data, and since the content of the files remains the same, no significant changes are made.

It's worth noting that the time column is excluded, as it is not part of the spectrogram.

🧠 1. Function process_spec(spec_id, split)

Saves the result in .npy format in a temporary folder (/tmp/dataset/hms-hbac), ready to be used during training.

In [ ]:

# Define a function to process a single eeg_id
def process_spec(spec_id, split="train"):
    spec_path = f"{BASE_PATH}/{split}_spectrograms/{spec_id}.parquet"
    spec = pd.read_parquet(spec_path)
    spec = spec.fillna(0).values[:, 1:].T # fill NaN values with 0, transpose for (Time, Freq) -> (Freq, Time)
    spec = spec.astype("float32")
    np.save(f"{SPEC_DIR}/{split}_spectrograms/{spec_id}.npy", spec)

🌀 2. Processing training data (train)

Retrieves all unique spectrogram IDs from the training data.

Uses Joblib to process all spectrograms in parallel (on all available CPU cores).

Displays a progress bar using tqdm.

In [ ]:
# Get unique spec_ids of train and valid data
spec_ids = df["spectrogram_id"].unique()

# Parallelize the processing using joblib for training data
_ = joblib.Parallel(n_jobs=-1, backend="loky")(
    joblib.delayed(process_spec)(spec_id, "train")
    for spec_id in tqdm(spec_ids, total=len(spec_ids))
)

🔬 3. Processing test data (test)

Same logic as above, but applied to test_df.

In [ ]:
# Get unique spec_ids of test data
test_spec_ids = test_df["spectrogram_id"].unique()

# Parallelize the processing using joblib for test data
_ = joblib.Parallel(n_jobs=-1, backend="loky")( #uses all available CPUs (n_jobs=-1)
    joblib.delayed(process_spec)(spec_id, "test")
    for spec_id in tqdm(test_spec_ids, total=len(test_spec_ids))
)

🍚 | DataLoader

This DataLoader first reads npy spectrogram files and extracts labeled subsamples using specified offset values. Then, it converts the spectrogram data into log spectrogram and applies the popular signal augmentation MixUp.

Note that, we are converting the mono channel signal to a 3-channel signal for using "ImageNet" weights of pretrained model.

build_augmenter(): Defines data augmentation transformations (spectrogram type).

MixUp: Mixes two images and their labels (useful for regularization).

Frequency masking: Cuts vertical bands (along the frequency), useful for robustness.

Time masking: Cuts horizontal bands (along the time), to simulate noise or losses.

👉 At each call, one or more of these maskings are applied with a probability of 50%.

In [ ]:
def build_augmenter(dim=CFG.image_size):
    augmenters = [
        keras_cv.layers.MixUp(alpha=2.0),
        keras_cv.layers.RandomCutout(height_factor=(1.0, 1.0),
                                     width_factor=(0.06, 0.1)), # freq-masking
        keras_cv.layers.RandomCutout(height_factor=(0.06, 0.1),
                                     width_factor=(1.0, 1.0)), # time-masking
    ]
    
    def augment(img, label):
        data = {"images":img, "labels":label}
        for augmenter in augmenters:
            if tf.random.uniform([]) < 0.5:
                data = augmenter(data, training=True)
        return data["images"], data["labels"]
    
    return augment

build_decoder(): reads an .npy file, transforms it (log + normalization), and prepares the labels.

decode_signal(path, offset=None)
* Reads the .npy file (in binary form).

* Removes a header (depending on the type).

* Reshapes to [400, T] (frequency x time representation).

* If offset is given, extracts a subset of the spectrogram (centered on an event).

* Switches to logarithmic scale (log), while clipping to avoid log(0).

* Then normalizes: subtracts the mean and divides by the standard deviation.

* Finally, converts to 3 RGB channels (via tile) to be able to use a pre-trained model on ImageNet.

decode_label(label)
* Converts a label to one-hot encoding.

decode_with_labels(path, offset, label)
* Combines the two functions above for a complete example.

In [ ]:
def build_decoder(with_labels=True, target_size=CFG.image_size, dtype=32):
    def decode_signal(path, offset=None):
        # Read .npy files and process the signal
        file_bytes = tf.io.read_file(path)
        sig = tf.io.decode_raw(file_bytes, tf.float32)
        sig = sig[1024//dtype:]  # Remove header tag
        sig = tf.reshape(sig, [400, -1])
        
        # Extract labeled subsample from full spectrogram using "offset"
        if offset is not None: 
            offset = offset // 2  # Only odd values are given
            sig = sig[:, offset:offset+300]
            
            # Pad spectrogram to ensure the same input shape of [400, 300]
            pad_size = tf.math.maximum(0, 300 - tf.shape(sig)[1])
            sig = tf.pad(sig, [[0, 0], [0, pad_size]])
            sig = tf.reshape(sig, [400, 300])
        
        # Log spectrogram 
        sig = tf.clip_by_value(sig, tf.math.exp(-4.0), tf.math.exp(8.0)) # avoid 0 in log
        sig = tf.math.log(sig)
        
        # Normalize spectrogram
        sig -= tf.math.reduce_mean(sig)
        sig /= tf.math.reduce_std(sig) + 1e-6
        
        # Mono channel to 3 channels to use "ImageNet" weights
        sig = tf.tile(sig[..., None], [1, 1, 3])
        return sig
    
    def decode_label(label):
        label = tf.one_hot(label, CFG.num_classes)
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [CFG.num_classes])
        return label
    
    def decode_with_labels(path, offset=None, label=None):
        sig = decode_signal(path, offset)
        label = decode_label(label)
        return (sig, label)
    
    return decode_with_labels if with_labels else decode_signal

build_dataset() : Builds a tf.data.Dataset pipeline with decoding, batching, shuffle, augmentation, etc.


Builds an optimized tf.data.Dataset from paths, offsets, and labels.

Steps:

* from_tensor_slices(...) : Builds the raw dataset.

* .map(decode_fn) : Decodes the files and prepares the labels.

* cache() : Cache in RAM or a folder to speed up iterations.

* repeat() : Repeats the data indefinitely (useful for model.fit() without a fixed epoch).

* shuffle(...) : Shuffles the data (important for training).

* batch(...) : Shapes the batches.

* .map(augment_fn) : Applies augmentation (if enabled).

* .prefetch(...) : Prefetch subsequent batches (improves performance).

In [ ]:
def build_dataset(paths, offsets=None, labels=None, batch_size=32, cache=True,
                  decode_fn=None, augment_fn=None,
                  augment=False, repeat=True, shuffle=1024, 
                  cache_dir="", drop_remainder=False):
    if cache_dir != "" and cache is True:
        os.makedirs(cache_dir, exist_ok=True)
    
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)
    
    if augment_fn is None:
        augment_fn = build_augmenter()
    
    AUTO = tf.data.experimental.AUTOTUNE
    slices = (paths, offsets) if labels is None else (paths, offsets, labels)
    
    ds = tf.data.Dataset.from_tensor_slices(slices) #builds the raw dataset.
    ds = ds.map(decode_fn, num_parallel_calls=AUTO) #decodes the files and prepares the labels.
    ds = ds.cache(cache_dir) if cache else ds  #caching in RAM or in a folder to speed up iterations.
    ds = ds.repeat() if repeat else ds  #repeats the data indefinitely (useful for model.fit() without fixed epoch).
    if shuffle: 
        ds = ds.shuffle(shuffle, seed=CFG.seed)  #mixes the data (important for learning).
        opt = tf.data.Options()
        opt.experimental_deterministic = False
        ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)  #forms batches.
    ds = ds.map(augment_fn, num_parallel_calls=AUTO) if augment else ds #applies the increase (if enabled).
    ds = ds.prefetch(AUTO)  #preloads subsequent batches (improves performance).
    return ds


🔪 | Data Split

In the following code snippet, the data is divided into 5 folds. Note that, the groups argument is used to prevent any overlap of patients between the training and validation sets, thus avoiding potential data leakage issues. Additionally, each split is stratified based on the class_label, ensuring a uniform distribution of class labels in each fold.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed) #Create a splitter for 5 folds, with a shuffle to mix the data and a random_state to make it reproducible.

df["fold"] = -1 #Initializes a fold column to -1 (to mark everything as "unassigned").

df.reset_index(drop=True, inplace=True) #Reinitializes the DataFrame indexes properly.
for fold, (train_idx, valid_idx) in enumerate(
    sgkf.split(df, y=df["class_label"], groups=df["patient_id"]) #Allows for stratification (balancing classes).
                                                                 #Ensures that EEGs from the same patient are in only one fold at a time.
):
    df.loc[valid_idx, "fold"] = fold
df.groupby(["fold", "class_name"])[["eeg_id"]].count().T

Build Train & Valid Dataset

Only first sample for each spectrogram_id is used in order to keep the dataset size manageable. Feel free to train on full data.

This code prepares the training and validation datasets for model training using a specific fold from a cross-validation split. It first samples one row per spectrogram (to avoid duplicates) and separates the data into training and validation sets based on the selected fold (CFG.fold). The build_dataset function is then used to construct TensorFlow datasets: the training dataset is set to repeat, shuffle, and apply data augmentation for robust learning, while the validation dataset is kept static (no repetition, no shuffling, no augmentation) to ensure reliable evaluation.

In [ ]:
# Sample from full data
sample_df = df.groupby("spectrogram_id").head(1).reset_index(drop=True)
train_df = sample_df[sample_df.fold != CFG.fold]
valid_df = sample_df[sample_df.fold == CFG.fold]
print(f"# Num Train: {len(train_df)} | Num Valid: {len(valid_df)}")

# Train
train_paths = train_df.spec2_path.values
train_offsets = train_df.spectrogram_label_offset_seconds.values.astype(int)
train_labels = train_df.class_label.values
train_ds = build_dataset(train_paths, train_offsets, train_labels, batch_size=CFG.batch_size,
                         repeat=True, shuffle=True, augment=True, cache=True)

# Valid
valid_paths = valid_df.spec2_path.values
valid_offsets = valid_df.spectrogram_label_offset_seconds.values.astype(int)
valid_labels = valid_df.class_label.values
valid_ds = build_dataset(valid_paths, valid_offsets, valid_labels, batch_size=CFG.batch_size,
                         repeat=False, shuffle=False, augment=False, cache=True)


Dataset Check

Let's visualize some samples from the dataset.

In [ ]:
imgs, tars = next(iter(train_ds))

num_imgs = 8
plt.figure(figsize=(4*4, num_imgs//4*5))
for i in range(num_imgs):
    plt.subplot(num_imgs//4, 4, i + 1)
    img = imgs[i].numpy()[...,0]  # Adjust as per your image data format
    img -= img.min()
    img /= img.max() + 1e-4
    tar = CFG.label2name[np.argmax(tars[i].numpy())]
    plt.imshow(img)
    plt.title(f"Target: {tar}")
    plt.axis('off')
    
plt.tight_layout()
plt.show()

🔍 | Loss & Metric



In [ ]:
LOSS = keras.losses.KLDivergence()

🤖 | Modeling

This notebook uses the EfficientNetV2 B2 from KerasCV's collection of pretrained models. To explore other models, simply modify the preset in the CFG (config). Check the KerasCV website for a list of available pretrained models.

In [ ]:
# Build Classifier
model = keras_cv.models.ImageClassifier.from_preset(
    CFG.preset, num_classes=CFG.num_classes
)

# Compile the model  
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
              loss=LOSS)

# Model Sumamry
model.summary()

⚓ | LR Schedule

A well-structured learning rate schedule is essential for efficient model training, ensuring optimal convergence and avoiding issues such as overshooting or stagnation.

This code defines a custom learning rate scheduler for training deep learning models using Keras. The function get_lr_callback creates a learning rate callback that dynamically adjusts the learning rate over training epochs based on different strategies, such as cosine decay, exponential decay, or step decay. It begins with a linear warm-up phase (lr_ramp_ep), can optionally sustain a fixed learning rate for a few epochs (lr_sus_ep), and then applies the selected decay strategy. In the cosine mode, for example, the learning rate gradually decreases following a cosine curve, promoting smoother convergence during training. Optionally, if plot=True, the learning rate schedule can be visualized. The function returns a LearningRateScheduler callback that can be passed to model.fit() to automatically adjust the learning rate during training.

In [ ]:
import math

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 5e-5, 6e-6 * batch_size, 1e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback


In [ ]:
lr_cb = get_lr_callback(CFG.batch_size, mode=CFG.lr_mode, plot=True)

Weights & Biasis config

In [ ]:
import wandb
wandb.login()   # demandera la clé, ou la lira dans la variable WANDB_API_KEY

In [ ]:
# Iterator for model run count
i=40

In [ ]:
import wandb
# Initialize W&B *before* tout entraînement ou accès à wandb.config
wandb.init(
    project="Harmful_Brain_Activity_Classification_with_KerasCV_and_Keras",
    entity="emnakraiem18-ek",
    config={  # Utilise les attributs de CFG
        "learning_rate": 1e-4,
        "epochs": CFG.epochs,
        "batch_size": CFG.batch_size,
        "image_size": CFG.image_size,
        "preset": CFG.preset,
        "lr_mode": CFG.lr_mode,
        "num_classes": CFG.num_classes,
        "fold": CFG.fold
    },
    name=f"Harmful_run-{i}"
)

# (optionnel) Accès à wandb.config si tu veux l’utiliser ensuite
CFG.batch_size = wandb.config.batch_size
CFG.epochs = wandb.config.epochs

💾 | Model Checkpointing

In [ ]:
ckpt_cb = keras.callbacks.ModelCheckpoint("best_model.keras",
                                         monitor='val_loss',
                                         save_best_only=True,
                                         save_weights_only=False,
                                         mode='min')

🚂 | Training

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
# Exemple de callbacks que tu aurais définis avant
checkpoint = ModelCheckpoint("best_model.keras", save_best_only=True, monitor='val_loss')
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

# Entraînement
history = model.fit(
    train_ds,
    steps_per_epoch=len(train_df) // CFG.batch_size,
    epochs=CFG.epochs,
    validation_data=valid_ds,
    callbacks=[
        WandbMetricsLogger(),                     # ✅ Nouveau logger W&B
        WandbModelCheckpoint("model.keras"),      # ✅ Sauvegarde W&B (optionnelle)
        checkpoint,                                # Sauvegarde locale du meilleur modèle
        early_stop                                 # Arrêt anticipé
    ],
    verbose=CFG.verbose
)


In [ ]:
# Afficher les clés disponibles dans history
print(history.history.keys())

# Courbe de loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Courbe de perte (Loss)')
plt.legend()

# Courbe d'accuracy (si disponible)
if 'accuracy' in history.history:
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Courbe de précision (Accuracy)')
    plt.legend()

plt.tight_layout()
plt.show()

🧪 | Prediction

Load Best Model

In [ ]:
model.load_weights("best_model.keras")

Build Test Dataset

In [ ]:
test_paths = test_df.spec2_path.values
test_ds = build_dataset(test_paths, batch_size=min(CFG.batch_size, len(test_df)),
                         repeat=False, shuffle=False, cache=False, augment=False)

Inference

In [ ]:
preds = model.predict(test_ds)

In [ ]:
print(test_df.columns)


📩 | Submission

This code block prepares the model's predictions for submission in the format required by the Kaggle competition. It starts by creating a copy of the test_df DataFrame, keeping only the eeg_id column, and then adds the model's predictions (preds) to it as columns named after classes (seizure_vote, lpd_vote, etc.). Next, it reads the sample_submission.csv file to ensure the format is respected, keeping only the eeg_id column, and then merges this file with the previous predictions based on eeg_id. Finally, the resulting file is exported as submission.csv, ready for submission to Kaggle.

In [ ]:
pred_df = test_df[["eeg_id"]].copy()
target_cols = [x.lower()+'_vote' for x in CFG.class_names]
pred_df[target_cols] = preds.tolist()

sub_df = pd.read_csv(f'{BASE_PATH}/sample_submission.csv')
sub_df = sub_df[["eeg_id"]].copy()
sub_df = sub_df.merge(pred_df, on="eeg_id", how="left")
sub_df.to_csv("submission.csv", index=False)
sub_df.head()